# Implémentation d'article : A Novel Text Mining Approach Based on TF-IDF and Support Vector Machine for News Classification

## Importation des librairies

In [56]:
import os
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import NuSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## Etape 1 : Pré-traitement du text 

### Importation du dataset

In [8]:
#fonction pour importer les librairies
def load_bbc_dataset(dataset_path):
    data = []
    labels = []

    # Parcourt chaque dossier
    for category in os.listdir(dataset_path):
        category_path = os.path.join(dataset_path, category)
        if os.path.isdir(category_path):
            for filename in os.listdir(category_path):
                file_path = os.path.join(category_path, filename)
                with open(file_path, 'r', encoding='latin1') as file:
                    content = file.read()
                    data.append(content)
                    labels.append(category)

    # Création du DataFrame
    df = pd.DataFrame({'text': data, 'label': labels})
    return df

In [32]:
dataset_path = r"C:\Users\wiame\Documents\1_SEMESTER_8\NLP\Projet\BBC dataset"
df = load_bbc_dataset(dataset_path)
df

,text,label
0,Ad sales boost Time Warner profit\n\nQuarterly...,business
1,Dollar gains on Greenspan speech\n\nThe dollar...,business
2,Yukos unit buyer faces loan claim\n\nThe owner...,business
3,High fuel prices hit BA's profits\n\nBritish A...,business
4,Pernod takeover talk lifts Domecq\n\nShares in...,business
...,...,...
2220,BT program to beat dialler scams\n\nBT is intr...,tech
2221,Spam e-mails tempt net shoppers\n\nComputer us...,tech
2222,Be careful how you code\n\nA new European dire...,tech
2223,US cyber security chief resigns\n\nThe man mak...,tech


### Elimination de la ponctuation, dates, phrases non nécessaire..etc

In [34]:
def clean_text(text):
    # 0. Supprimer les retours à la ligne
    text = text.replace('\n', ' ').replace('\r', ' ')

    # 1. Supprimer les citations
    text = re.sub(r'["“”‘’].*?["“”‘’]', '', text)

    # 2. Supprimer les dates
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '', text)
    text = re.sub(r'\b(19|20)\d{2}\b', '', text)

    # 3. Supprimer les phrases trop courtes
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s for s in sentences if len(s.split()) >= 3]
    text = ' '.join(sentences)

    # 4. Supprimer la ponctuation résiduelle
    text = re.sub(r'[!\"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~]', ' ', text)

    # 5. Supprimer les chiffres
    text = re.sub(r'\d+', '', text)

    # 6. Nettoyer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [36]:
df['cleaned_text'] = df['text'].apply(clean_text)
df

,text,label,cleaned_text
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,Ad sales boost Time Warner profit Quarterly pr...
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,Dollar gains on Greenspan speech The dollar ha...
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,Yukos unit buyer faces loan claim The owners o...
3,High fuel prices hit BA's profits\n\nBritish A...,business,High fuel prices hit BA's profits British Airw...
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,Pernod takeover talk lifts Domecq Shares in UK...
...,...,...,...
2220,BT program to beat dialler scams\n\nBT is intr...,tech,BT program to beat dialler scams BT is introdu...
2221,Spam e-mails tempt net shoppers\n\nComputer us...,tech,Spam e-mails tempt net shoppers Computer users...
2222,Be careful how you code\n\nA new European dire...,tech,Be careful how you code A new European directi...
2223,US cyber security chief resigns\n\nThe man mak...,tech,US cyber security chief resigns The man making...


### Transformation des majuscules

In [38]:
df['cleaned_text'] = df['cleaned_text'].apply(lambda x: x.lower())
df

,text,label,cleaned_text
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,ad sales boost time warner profit quarterly pr...
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,dollar gains on greenspan speech the dollar ha...
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,yukos unit buyer faces loan claim the owners o...
3,High fuel prices hit BA's profits\n\nBritish A...,business,high fuel prices hit ba's profits british airw...
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,pernod takeover talk lifts domecq shares in uk...
...,...,...,...
2220,BT program to beat dialler scams\n\nBT is intr...,tech,bt program to beat dialler scams bt is introdu...
2221,Spam e-mails tempt net shoppers\n\nComputer us...,tech,spam e-mails tempt net shoppers computer users...
2222,Be careful how you code\n\nA new European dire...,tech,be careful how you code a new european directi...
2223,US cyber security chief resigns\n\nThe man mak...,tech,us cyber security chief resigns the man making...


### Tokenization

In [40]:
df['tokens'] = df['cleaned_text'].apply(word_tokenize)
df

,text,label,cleaned_text,tokens
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,ad sales boost time warner profit quarterly pr...,"[ad, sales, boost, time, warner, profit, quart..."
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,dollar gains on greenspan speech the dollar ha...,"[dollar, gains, on, greenspan, speech, the, do..."
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,yukos unit buyer faces loan claim the owners o...,"[yukos, unit, buyer, faces, loan, claim, the, ..."
3,High fuel prices hit BA's profits\n\nBritish A...,business,high fuel prices hit ba's profits british airw...,"[high, fuel, prices, hit, ba, 's, profits, bri..."
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,pernod takeover talk lifts domecq shares in uk...,"[pernod, takeover, talk, lifts, domecq, shares..."
...,...,...,...,...
2220,BT program to beat dialler scams\n\nBT is intr...,tech,bt program to beat dialler scams bt is introdu...,"[bt, program, to, beat, dialler, scams, bt, is..."
2221,Spam e-mails tempt net shoppers\n\nComputer us...,tech,spam e-mails tempt net shoppers computer users...,"[spam, e-mails, tempt, net, shoppers, computer..."
2222,Be careful how you code\n\nA new European dire...,tech,be careful how you code a new european directi...,"[be, careful, how, you, code, a, new, european..."
2223,US cyber security chief resigns\n\nThe man mak...,tech,us cyber security chief resigns the man making...,"[us, cyber, security, chief, resigns, the, man..."


### Elimination des stopwords

In [45]:
stop_words = set(stopwords.words('english'))
def remove_stopwords_from_tokens(tokens):
    return [word for word in tokens if word.lower() not in stop_words]

df['tokens'] = df['tokens'].apply(remove_stopwords_from_tokens)
df

,text,label,cleaned_text,tokens
0,Ad sales boost Time Warner profit\n\nQuarterly...,business,ad sales boost time warner profit quarterly pr...,"[ad, sales, boost, time, warner, profit, quart..."
1,Dollar gains on Greenspan speech\n\nThe dollar...,business,dollar gains on greenspan speech the dollar ha...,"[dollar, gains, greenspan, speech, dollar, hit..."
2,Yukos unit buyer faces loan claim\n\nThe owner...,business,yukos unit buyer faces loan claim the owners o...,"[yukos, unit, buyer, faces, loan, claim, owner..."
3,High fuel prices hit BA's profits\n\nBritish A...,business,high fuel prices hit ba's profits british airw...,"[high, fuel, prices, hit, ba, 's, profits, bri..."
4,Pernod takeover talk lifts Domecq\n\nShares in...,business,pernod takeover talk lifts domecq shares in uk...,"[pernod, takeover, talk, lifts, domecq, shares..."
...,...,...,...,...
2220,BT program to beat dialler scams\n\nBT is intr...,tech,bt program to beat dialler scams bt is introdu...,"[bt, program, beat, dialler, scams, bt, introd..."
2221,Spam e-mails tempt net shoppers\n\nComputer us...,tech,spam e-mails tempt net shoppers computer users...,"[spam, e-mails, tempt, net, shoppers, computer..."
2222,Be careful how you code\n\nA new European dire...,tech,be careful how you code a new european directi...,"[careful, code, new, european, directive, coul..."
2223,US cyber security chief resigns\n\nThe man mak...,tech,us cyber security chief resigns the man making...,"[us, cyber, security, chief, resigns, man, mak..."


## Etape 2 : Extaction des caractéristiques avec TF-IDF

In [71]:
#Initialiser le vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,1))

#Appliquer le vectorizer la colonne texte, TF-IDF attend une liste de chaînes de caractères, donc il faut les rejoindre en chaînes
X = vectorizer.fit_transform(df['tokens'].apply(lambda x: ' '.join(x)))

# Afficher les premiers mots extraits
print(X)
print('='*148)
print(X.shape)

  (0, 49)	0.0636121728749424
  (0, 3860)	0.15256213681912278
  (0, 518)	0.04374007548654625
  (0, 4542)	0.07448747966825062
  (0, 4836)	0.2712024452672184
  (0, 3440)	0.24769600374898407
  (0, 3531)	0.06250754871333473
  (0, 3442)	0.23669578734895735
  (0, 4722)	0.07132222094642292
  (0, 2771)	0.03982265487957723
  (0, 1905)	0.04213510365735286
  (0, 2406)	0.05820901429353397
  (0, 498)	0.16222450789428858
  (0, 4528)	0.05375220323066761
  (0, 2869)	0.0331204862702417
  (0, 1149)	0.03693729109721235
  (0, 4979)	0.05633078311618263
  (0, 1380)	0.07114833047347294
  (0, 1734)	0.03177578579276971
  (0, 3065)	0.04248453889262507
  (0, 454)	0.036186182963406976
  (0, 2311)	0.04788430680547678
  (0, 1942)	0.11192377292324107
  (0, 2079)	0.0651674679576661
  (0, 4179)	0.09604981563718208
  :	:
  (2224, 1858)	0.2204218244411353
  (2224, 9)	0.06232097160369254
  (2224, 2693)	0.06313120221939741
  (2224, 4963)	0.05520270308198838
  (2224, 2194)	0.0462381435082346
  (2224, 4706)	0.072141605778314

## Etape 3 : Entrainement avec SVM

In [73]:
#Extraction de la colonne label
y = df['label']

# 1. Séparer train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Initialiser le modèle NuSVC avec kernel RBF et nu=0.5
model = NuSVC(kernel='rbf', nu=0.5)

# 3. Entraîner
model.fit(X_train, y_train)

# 4. Prédire sur le test
y_pred = model.predict(X_test)

# 5. Évaluer
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9910112359550561
               precision    recall  f1-score   support

     business       1.00      0.98      0.99       102
entertainment       1.00      0.99      0.99        77
     politics       0.99      0.99      0.99        84
        sport       0.98      1.00      0.99       102
         tech       0.99      1.00      0.99        80

     accuracy                           0.99       445
    macro avg       0.99      0.99      0.99       445
 weighted avg       0.99      0.99      0.99       445

